# NTP dan SNTP — menjaga jam perangkat tetap jujur

Server NTP berreferensi GPS dan perangkat dengan jam yang melenceng berbagi jaringan dalam memori dengan 10 ms tiap arah. `SntpClient` yang sama mengukur komputer Anda terhadap `pool.ntp.org` dengan `UseServer("pool.ntp.org")`; ia tidak pernah mengubah jam sistem.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.19.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.19.0-preview.1"

## Cap waktu dan paket
NTP menghitung detik sejak 1900 dalam 32 bit ditambah pecahan 32 bit; detiknya berputar pada 2036 dan aturan era memetakannya kembali.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Ntp;
using IoTCom.Net.Transports;
using System.Net;

var unix = NtpTimestamp.FromDateTime(new DateTime(1970, 1, 1, 0, 0, 0, DateTimeKind.Utc));
Console.WriteLine($"1970-01-01 = 0x{unix.Raw:X16}");
Console.WriteLine($"0x0000000100000000 = {new NtpTimestamp(1UL << 32)}  (era 1)");
var request = new NtpPacket { Mode = NtpMode.Client, Transmit = NtpTimestamp.FromDateTime(DateTime.UtcNow) }.Encode();
foreach (var f in NtpPacket.Describe(request)) Console.WriteLine($"{f.Name,-11} {f.Value}");

## Satu pertukaran: offset dan delay
T1 meninggalkan perangkat, T2 tiba di server, T3 meninggalkan server, T4 tiba kembali.

In [ ]:
var ntpNet = new InMemoryDatagramNetwork { Latency = TimeSpan.FromMilliseconds(10) };
var ntpAddress = new IPEndPoint(IPAddress.Parse("10.0.0.1"), 123);
var gpsServer = NtpServer.Create(o => o.UseInMemory(ntpNet, ntpAddress).WithReference("GPS"));
await gpsServer.StartAsync();
var deviceClock = new DriftingClock(TimeSpan.FromSeconds(-3.2), driftPpm: 20_000);   // 2 %: exaggerated
var sntp = SntpClient.Create(o => { o.UseInMemory(ntpNet); o.UseServer(ntpAddress); o.Clock = () => deviceClock.UtcNow; o.MinimumPollInterval = TimeSpan.Zero; });
var first = await sntp.QueryAsync(ntpAddress);
Console.WriteLine($"T1 {first.T1}\nT2 {first.T2}\nT3 {first.T3}\nT4 {first.T4}");
Console.WriteLine($"offset {first.Offset.TotalMilliseconds:+0.0} ms, delay {first.RoundTripDelay.TotalMilliseconds:0.0} ms");

## Mendisiplinkan jam
Geser jam perangkat sebesar offset yang terukur dan lihat galatnya tetap kecil di antara sinkronisasi.

In [ ]:
deviceClock.Step(first.Offset);
for (var i = 0; i < 3; i++)
{
    await Task.Delay(500);
    var before = deviceClock.Error;
    var r = await sntp.QueryAsync(ntpAddress);
    deviceClock.Step(r.Offset);
    Console.WriteLine($"drifted {before.TotalMilliseconds:+0.0} ms, corrected to {deviceClock.Error.TotalMilliseconds:+0.0} ms");
}

## Saat server tidak bisa dipercaya
Server yang kehilangan referensinya mengiklankan leap 3; server yang sibuk menjawab kiss-o'-death RATE. Keduanya ditolak.

In [ ]:
gpsServer.Leap = NtpLeap.Unsynchronised;
try { await sntp.QueryAsync(ntpAddress); } catch (DeviceException e) { Console.WriteLine(e.Message); }
var busyAddress = new IPEndPoint(IPAddress.Parse("10.0.0.2"), 123);
var busy = NtpServer.Create(o => { o.UseInMemory(ntpNet, busyAddress); o.RateLimit = TimeSpan.FromSeconds(10); });
await busy.StartAsync();
await sntp.QueryAsync(busyAddress);
try { await sntp.QueryAsync(busyAddress); } catch (NtpKissOfDeathException e) { Console.WriteLine($"{e.Code}: {e.Message}"); }
await sntp.DisposeAsync();
await busy.DisposeAsync();
await gpsServer.DisposeAsync();

## Lebih lanjut
`iotcom ntp query`, `iotcom ntp query --sim --frames`, `iotcom ntp serve`, *Sinkronisasi jam armada lewat NTP* di Gallery, dan sampel NtpClock. Lihat `docs/id/protocols/ntp.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*